In [69]:
import rasterio as rio
import rioxarray as rxr
from rasterio.warp import calculate_default_transform, reproject, Resampling


In [71]:
#file paths
QSI_path = "SNEX20_QSI_VH_0.5M_USIDMC_20210315_20210315.tif"
GEDI_path = "Raw/Forest_height_2019_NAM.tif"

QSI = rxr.open_rasterio(QSI_path, masked=True).squeeze()
GEDI = rxr.open_rasterio(GEDI_path, masked=True).squeeze()

print(QSI.rio.crs)
print(GEDI.rio.crs)


EPSG:6340
EPSG:4326


In [72]:
QSI_proj = QSI.rio.reproject(GEDI.rio.crs)
print(QSI_proj.rio.crs)
print(GEDI.rio.crs)


EPSG:4326
EPSG:4326


In [77]:
minx, miny, maxx, maxy = QSI_proj.rio.bounds()
print(minx, miny, maxx, maxy)

-115.73613833978959 43.89832331365237 -115.6219197023722 43.994132352188025


In [78]:
GEDI_crop = GEDI.rio.clip_box(
    minx=minx,
    miny=miny,
    maxx=maxx,
    maxy=maxy
)

In [80]:
print(GEDI_crop.rio.bounds())
print(QSI_proj.rio.bounds())

(-115.73624999999998, 43.898250000000004, -115.62174999999999, 43.99425)
(-115.73613833978959, 43.89832331365237, -115.6219197023722, 43.994132352188025)


In [84]:
print(GEDI.rio.resolution())
print(GEDI_crop.rio.resolution())
print(QSI_proj.rio.resolution())

(0.00025, -0.00025)
(0.00024999999999999654, -0.0002500000000000064)
(5.274956699644675e-06, -5.274956699644675e-06)


In [87]:
QSI_resample = QSI_proj.rio.reproject(QSI_proj.rio.crs,resolution = GEDI_crop.rio.resolution())

In [88]:
print(GEDI_crop.rio.resolution())
print(QSI_resample.rio.resolution())

(0.00024999999999999654, -0.0002500000000000064)
(0.00024999999999999654, -0.0002500000000000064)


In [89]:
GEDI_resample = GEDI_crop.rio.reproject_match(QSI_resample, resampling=Resampling.bilinear)